# 02 - Model TrainingTrain a binary classification model on the reference dataset.- Train a RandomForestClassifier- Compute classification metrics- Save model artifact and metadata with joblib

In [ ]:
import numpy as npimport pandas as pdimport plotly.express as pxfrom pathlib import Pathfrom src.data.data_loader import load_reference_data, load_config, split_features_targetfrom src.model.train import train_model, evaluate_on_split, save_artifactsfrom sklearn.model_selection import train_test_splitconfig = load_config()df = load_reference_data(config)print(f"Reference data shape: {df.shape}")

In [ ]:
target_column = config["data"]["target_column"]X, y = split_features_target(df, target_column=target_column)# Keep only numeric featuresX = X.select_dtypes(include=["number"])print(f"Feature columns: {list(X.columns)}")print(f"Target distribution:\n{y.value_counts(normalize=True)}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42, stratify=y)print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
model_config = config.get("model", {})model = train_model(X_train.values, y_train.values, config=model_config)print("Model trained successfully.")

In [ ]:
metrics = evaluate_on_split(model, X_test.values, y_test.values)print("Test set metrics:")for k, v in metrics.items():    print(f"  {k}: {v:.4f}")

In [ ]:
save_artifacts(model, list(X.columns), metrics, config=config)print("Artifacts saved to models/")

In [ ]:
import joblibfrom pathlib import Pathmodel_path = Path(config["paths"]["model_artifact"])metadata_path = Path(config["paths"]["model_metadata"])loaded_model = joblib.load(model_path)import jsonwith open(metadata_path) as f:    metadata = json.load(f)print(f"Model type: {metadata['model_type']}")print(f"Features: {metadata['n_features']}")print(f"Trained at: {metadata['trained_at']}")print(f"Test accuracy: {metadata['metrics']['accuracy']:.4f}")

## Key Observations- The model achieves strong performance on the held-out test set.- Model artifact and metadata are persisted for use in prediction and monitoring.- The pipeline is reproducible via `python scripts/run_pipeline.py`.